# 02 — Stage A (disease anchors) and Stage B (reference network) (Sections 3.3–3.4)

In [ ]:
import os
from pathlib import Path

# Folder that holds the downloaded public data; all outputs are written into it.
# Set the environment variable GBM_DATA_DIR, or edit the default below.
BASE = Path(os.environ.get("GBM_DATA_DIR", Path.home() / "Downloads"))
print("Data folder:", BASE)

### STAGE A — FINAL DIFFERENTIAL EXPRESSION ANALYSIS

In [ ]:
# ======================================================================
# STAGE A — FINAL DIFFERENTIAL EXPRESSION ANALYSIS
# ======================================================================
#
# DISCOVERY COHORT:
#
#   60 independent TCGA-GBM primary tumors
#   334 unique GTEx cortical normal donors
#
# GTEx regions:
#   - Brain - Cortex
#   - Brain - Frontal Cortex (BA9)
#
# IMPORTANT:
#
# These 60 GBM patients are NOT part of the main 144-patient
# survival cohort.
#
# Stage A is ONLY for independent gene discovery.
#
#
# PIPELINE:
#
# recount3 / Monorail
#        ↓
# raw gene coverage (gene_sums)
#        ↓
# coverage / average mapped read length
#        ↓
# estimated read counts
#        ↓
# low-expression filtering
#        ↓
# PyDESeq2
#        ↓
# log2FC + p-value + FDR
#        ↓
# FDR < 0.05 AND |log2FC| >= 1.5
#
# ======================================================================


# ======================================================================
# 0 — IMPORTS
# ======================================================================

import gzip
import io
import math
from pathlib import Path

import numpy as np
import pandas as pd
import requests


# ======================================================================
# PyDESeq2
# ======================================================================

try:

    from pydeseq2.dds import DeseqDataSet
    from pydeseq2.ds import DeseqStats

except ImportError:

    raise ImportError(
        "\nPyDESeq2 is not installed.\n\n"
        "Run in another Jupyter cell:\n\n"
        "%pip install pydeseq2\n\n"
        "Restart the kernel and run this script again."
    )


# ======================================================================
# 1 — PATHS
# ======================================================================

DOWNLOADS = BASE

STAGE_A_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_A_DISCOVERY"
)


TUMOR_SAMPLE_FILE = (
    STAGE_A_DIR
    / "STAGE_A_TCGA_GBM_INDEPENDENT_TUMORS.csv"
)


# IMPORTANT:
# this is the FINAL 334-control file
NORMAL_SAMPLE_FILE = (
    STAGE_A_DIR
    / "STAGE_A_GTEX_CORTEX_FINAL_QC.csv"
)


OUTDIR = (
    STAGE_A_DIR
    / "DE_RESULTS"
)

OUTDIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# 2 — RECOUNT3 CONFIGURATION
# ======================================================================

BASE = (
    "http://duffel.rail.bio/"
    "recount3/human/data_sources"
)

TCGA_SOURCE = "tcga"
TCGA_PROJECT = "GBM"

GTEX_SOURCE = "gtex"
GTEX_PROJECT = "BRAIN"


# ======================================================================
# 3 — DE SETTINGS
# ======================================================================

FDR_THRESHOLD = 0.05

LOG2FC_THRESHOLD = 1.5


# low expression filtering
MIN_COUNT = 10

# >=20% of smaller group
MIN_GROUP_FRACTION = 0.20


CHUNK_SIZE = 5000

SAVE_FILTERED_COUNTS = True


# ======================================================================
# 4 — HELPER FUNCTIONS
# ======================================================================

def section(title):

    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)


def project_shard(project):

    return project[-2:]


def clean_string(value):

    if pd.isna(value):
        return None

    value = str(value).strip()

    if value == "":
        return None

    if value.lower() == "nan":
        return None

    return value


# ======================================================================
# URL BUILDERS
# ======================================================================

def metadata_url(source, project):

    shard = project_shard(project)

    return (
        f"{BASE}/{source}/metadata/"
        f"{shard}/{project}/"
        f"{source}.{source}.{project}.MD.gz"
    )


def qc_url(source, project):

    shard = project_shard(project)

    return (
        f"{BASE}/{source}/metadata/"
        f"{shard}/{project}/"
        f"{source}.recount_qc.{project}.MD.gz"
    )


def gene_sums_url(source, project):

    shard = project_shard(project)

    return (
        f"{BASE}/{source}/gene_sums/"
        f"{shard}/{project}/"
        f"{source}.gene_sums.{project}.G026.gz"
    )


# ======================================================================
# DOWNLOAD SMALL METADATA/QC TABLE
# ======================================================================

def fetch_gz_tsv(url, timeout=300):

    print("Downloading:")
    print(url)

    response = requests.get(
        url,
        timeout=timeout
    )

    response.raise_for_status()

    with gzip.open(
        io.BytesIO(response.content),
        "rt"
    ) as handle:

        dataframe = pd.read_csv(
            handle,
            sep="\t",
            low_memory=False
        )

    return dataframe


# ======================================================================
# IMPORTANT FIX:
# READ REAL gene_sums HEADER
#
# recount3 files start with metadata lines such as:
#
# ##annotation=G026
#
# These are NOT the table header.
# ======================================================================

def get_gz_header(url, timeout=300):

    with requests.get(
        url,
        stream=True,
        timeout=timeout
    ) as response:

        response.raise_for_status()

        response.raw.decode_content = False


        with gzip.GzipFile(
            fileobj=response.raw,
            mode="rb"
        ) as gz:

            while True:

                raw_line = gz.readline()

                if not raw_line:

                    raise RuntimeError(
                        "End of gene_sums file reached "
                        "before finding the actual header."
                    )


                line = (
                    raw_line
                    .decode("utf-8")
                    .rstrip("\r\n")
                )


                # ------------------------------------------------------
                # Ignore recount3 metadata/comment lines
                # ------------------------------------------------------

                if line.startswith("##"):

                    print(
                        "Skipping metadata line:",
                        line
                    )

                    continue


                # ------------------------------------------------------
                # First non-comment line should be actual TSV header
                # ------------------------------------------------------

                header = line.split("\t")


                if len(header) < 2:

                    raise RuntimeError(
                        "Unexpected gene_sums header:\n"
                        f"{line}"
                    )


                return header


# ======================================================================
# 5 — LOAD FINAL FROZEN SAMPLE LISTS
# ======================================================================

section(
    "1 — FINAL FROZEN STAGE-A SAMPLE LISTS"
)


tumor_samples = pd.read_csv(
    TUMOR_SAMPLE_FILE,
    low_memory=False
)


normal_samples = pd.read_csv(
    NORMAL_SAMPLE_FILE,
    low_memory=False
)


print("Tumor file:")
print(TUMOR_SAMPLE_FILE)

print("\nNormal file:")
print(NORMAL_SAMPLE_FILE)


# ======================================================================
# Tumor IDs
# ======================================================================

if "Recount sample ID" not in tumor_samples.columns:

    raise ValueError(
        "Tumor file does not contain "
        "'Recount sample ID'.\n\n"
        f"Columns:\n{tumor_samples.columns.tolist()}"
    )


tumor_ids = (
    tumor_samples["Recount sample ID"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)


# ======================================================================
# Normal IDs
# ======================================================================

possible_normal_columns = [

    "Recount sample ID",
    "external_id",
    "sample_id",
    "Sample ID"
]


normal_id_col = None


for column in possible_normal_columns:

    if column in normal_samples.columns:

        normal_id_col = column
        break


if normal_id_col is None:

    raise ValueError(
        "Could not identify GTEx sample ID column.\n\n"
        f"Columns:\n{normal_samples.columns.tolist()}"
    )


print(
    "\nNormal sample ID column used:",
    normal_id_col
)


normal_ids = (
    normal_samples[normal_id_col]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)


print(
    "\nFrozen tumors:",
    len(tumor_ids)
)

print(
    "Frozen normals:",
    len(normal_ids)
)

print(
    "Unique tumor IDs:",
    len(set(tumor_ids))
)

print(
    "Unique normal IDs:",
    len(set(normal_ids))
)


# ======================================================================
# SAFETY CHECKS
# ======================================================================

if len(tumor_ids) != 60:

    raise RuntimeError(
        f"Expected 60 tumors, "
        f"found {len(tumor_ids)}."
    )


if len(normal_ids) != 334:

    raise RuntimeError(
        f"Expected 334 controls, "
        f"found {len(normal_ids)}."
    )


if len(set(tumor_ids)) != 60:

    raise RuntimeError(
        "Duplicate tumor IDs detected."
    )


if len(set(normal_ids)) != 334:

    raise RuntimeError(
        "Duplicate normal IDs detected."
    )


print(
    "\n✓ Final Stage-A cohort confirmed"
)

print(
    "60 tumors"
)

print(
    "334 normal cortical controls"
)


# ======================================================================
# 6 — RECOUNT3 PROJECT METADATA
# ======================================================================

section(
    "2 — RECOUNT3 PROJECT METADATA"
)


tcga_meta = fetch_gz_tsv(
    metadata_url(
        TCGA_SOURCE,
        TCGA_PROJECT
    )
)


gtex_meta = fetch_gz_tsv(
    metadata_url(
        GTEX_SOURCE,
        GTEX_PROJECT
    )
)


print(
    "\nTCGA metadata shape:",
    tcga_meta.shape
)

print(
    "GTEx metadata shape:",
    gtex_meta.shape
)


# ======================================================================
# 7 — RECOUNT3 QC
# ======================================================================

section(
    "3 — RECOUNT3 QC METADATA"
)


tcga_qc = fetch_gz_tsv(
    qc_url(
        TCGA_SOURCE,
        TCGA_PROJECT
    )
)


gtex_qc = fetch_gz_tsv(
    qc_url(
        GTEX_SOURCE,
        GTEX_PROJECT
    )
)


print(
    "\nTCGA QC shape:",
    tcga_qc.shape
)

print(
    "GTEx QC shape:",
    gtex_qc.shape
)


# ======================================================================
# 8 — AVERAGE MAPPED READ LENGTH
# ======================================================================

section(
    "4 — AVERAGE MAPPED READ LENGTH"
)


def find_average_length_column(df):

    # ----------------------------------------------------------
    # Preferred exact names
    # ----------------------------------------------------------

    preferred = [

        "recount_qc.star.average_mapped_length",

        "star.average_mapped_length"
    ]


    for column in preferred:

        if column in df.columns:

            print(
                "Using mapped-length column:",
                column
            )

            return column


    # ----------------------------------------------------------
    # fallback
    # explicitly exclude length2
    # ----------------------------------------------------------

    candidates = [

        c for c in df.columns

        if (
            "average_mapped_length"
            in c.lower()
        )

        and

        (
            "length2"
            not in c.lower()
        )
    ]


    if len(candidates) == 1:

        print(
            "Using mapped-length column:",
            candidates[0]
        )

        return candidates[0]


    if len(candidates) == 0:

        all_candidates = [

            c for c in df.columns

            if (
                "average_mapped_length"
                in c.lower()
            )
        ]


        raise ValueError(
            "No usable average mapped-length "
            "column found.\n"
            f"All candidates:\n{all_candidates}"
        )


    raise ValueError(
        "Still ambiguous mapped-length columns:\n"
        f"{candidates}"
    )


tcga_length_col = (
    find_average_length_column(
        tcga_qc
    )
)


gtex_length_col = (
    find_average_length_column(
        gtex_qc
    )
)


print(
    "TCGA length column:",
    tcga_length_col
)

print(
    "GTEx length column:",
    gtex_length_col
)


# ======================================================================
# 9 — EXPRESSION FILE URLS
# ======================================================================

tcga_gene_url = gene_sums_url(
    TCGA_SOURCE,
    TCGA_PROJECT
)


gtex_gene_url = gene_sums_url(
    GTEX_SOURCE,
    GTEX_PROJECT
)


# ======================================================================
# 10 — READ REAL EXPRESSION HEADERS
# ======================================================================

section(
    "5 — EXPRESSION FILE HEADERS"
)


tcga_header = get_gz_header(
    tcga_gene_url
)


gtex_header = get_gz_header(
    gtex_gene_url
)


print(
    "\nTCGA gene_sums columns:",
    len(tcga_header)
)


print(
    "GTEx gene_sums columns:",
    len(gtex_header)
)


print(
    "\nTCGA first 10 columns:"
)

print(
    tcga_header[:10]
)


print(
    "\nGTEx first 10 columns:"
)

print(
    gtex_header[:10]
)


tcga_gene_col = (
    tcga_header[0]
)

gtex_gene_col = (
    gtex_header[0]
)


print(
    "\nTCGA gene-ID column:",
    tcga_gene_col
)

print(
    "GTEx gene-ID column:",
    gtex_gene_col
)


# ======================================================================
# sanity check
# ======================================================================

if len(tcga_header) <= 2:

    raise RuntimeError(
        "TCGA gene_sums header still looks invalid."
    )


if len(gtex_header) <= 2:

    raise RuntimeError(
        "GTEx gene_sums header still looks invalid."
    )


# ======================================================================
# 11 — SAMPLE → EXPRESSION COLUMN MAPPING
# ======================================================================

section(
    "6 — SAMPLE → EXPRESSION COLUMN MAPPING"
)


def resolve_expression_mapping(
    frozen_ids,
    metadata,
    expression_header,
    source_name
):

    possible_meta_ids = [

        "external_id",

        "rail_id",

        "tcga_barcode"
    ]


    possible_meta_ids = [

        column

        for column in possible_meta_ids

        if column in metadata.columns
    ]


    if len(possible_meta_ids) == 0:

        raise RuntimeError(
            f"{source_name}: "
            "no usable metadata identifier columns."
        )


    expression_columns = set(
        expression_header[1:]
    )


    records = []


    for frozen_id in frozen_ids:

        frozen_id = (
            str(frozen_id)
            .strip()
        )


        row_mask = pd.Series(
            False,
            index=metadata.index
        )


        # ----------------------------------------------------------
        # Find frozen sample in metadata
        # ----------------------------------------------------------

        for column in possible_meta_ids:

            values = (
                metadata[column]
                .astype(str)
                .str.strip()
            )


            row_mask |= (
                values == frozen_id
            )


        matched_rows = (
            metadata[
                row_mask
            ]
        )


        if len(matched_rows) == 0:

            raise RuntimeError(
                f"{source_name}: "
                f"{frozen_id!r} "
                "not found in recount3 metadata."
            )


        expression_matches = []


        # ----------------------------------------------------------
        # Find identifier present in expression header
        # ----------------------------------------------------------

        for row_index, row in matched_rows.iterrows():

            for column in possible_meta_ids:

                value = clean_string(
                    row[column]
                )


                if value is None:

                    continue


                if value in expression_columns:

                    expression_matches.append(
                        {
                            "Frozen ID":
                                frozen_id,

                            "Expression column":
                                value,

                            "Matched using":
                                column,

                            "Metadata row":
                                row_index,

                            "external_id":
                                clean_string(
                                    row.get(
                                        "external_id"
                                    )
                                ),

                            "rail_id":
                                clean_string(
                                    row.get(
                                        "rail_id"
                                    )
                                ),

                            "tcga_barcode":
                                clean_string(
                                    row.get(
                                        "tcga_barcode"
                                    )
                                )
                        }
                    )


        # ----------------------------------------------------------
        # Remove duplicate mappings
        # ----------------------------------------------------------

        unique_mapping = {}


        for item in expression_matches:

            unique_mapping[
                item["Expression column"]
            ] = item


        expression_matches = list(
            unique_mapping.values()
        )


        if len(expression_matches) == 0:

            raise RuntimeError(
                f"{source_name}: "
                f"{frozen_id!r} exists in metadata "
                "but no corresponding expression "
                "column was found."
            )


        if len(expression_matches) > 1:

            raise RuntimeError(
                f"{source_name}: "
                f"{frozen_id!r} maps to multiple "
                "expression columns:\n"
                f"{expression_matches}"
            )


        records.append(
            expression_matches[0]
        )


    mapping = pd.DataFrame(
        records
    )


    # ----------------------------------------------------------
    # one-to-one mapping verification
    # ----------------------------------------------------------

    if (
        mapping[
            "Expression column"
        ]
        .duplicated()
        .any()
    ):

        duplicates = mapping[
            mapping[
                "Expression column"
            ]
            .duplicated(
                keep=False
            )
        ]


        raise RuntimeError(
            f"{source_name}: multiple samples mapped "
            "to the same expression column:\n"
            f"{duplicates}"
        )


    print(
        f"{source_name}: "
        f"{len(mapping)}/{len(frozen_ids)} "
        "samples mapped successfully."
    )


    print(
        "\nMapping identifier:"
    )


    print(
        mapping[
            "Matched using"
        ]
        .value_counts()
        .to_string()
    )


    return mapping


tcga_mapping = resolve_expression_mapping(

    tumor_ids,

    tcga_meta,

    tcga_header,

    "TCGA"
)


gtex_mapping = resolve_expression_mapping(

    normal_ids,

    gtex_meta,

    gtex_header,

    "GTEx"
)


# ======================================================================
# Save mapping audit
# ======================================================================

tcga_mapping.to_csv(

    OUTDIR
    / "STAGE_A_TCGA_EXPRESSION_MAPPING.csv",

    index=False
)


gtex_mapping.to_csv(

    OUTDIR
    / "STAGE_A_GTEX_EXPRESSION_MAPPING.csv",

    index=False
)


# ======================================================================
# 12 — MAP EXPRESSION SAMPLES TO QC READ LENGTH
# ======================================================================

section(
    "7 — QC / READ-LENGTH MAPPING"
)


def get_sample_lengths(
    mapping,
    qc,
    length_column,
    source_name
):

    possible_qc_ids = [

        "external_id",

        "rail_id",

        "tcga_barcode"
    ]


    possible_qc_ids = [

        c for c in possible_qc_ids

        if c in qc.columns
    ]


    if len(possible_qc_ids) == 0:

        raise RuntimeError(
            f"{source_name}: "
            "no usable QC identifier columns."
        )


    lengths = {}

    audit_rows = []


    for _, map_row in mapping.iterrows():

        frozen_id = (
            map_row[
                "Frozen ID"
            ]
        )


        expression_column = (
            map_row[
                "Expression column"
            ]
        )


        identifiers = set()


        for key in [

            "Frozen ID",

            "Expression column",

            "external_id",

            "rail_id",

            "tcga_barcode"
        ]:

            value = clean_string(
                map_row.get(key)
            )


            if value is not None:

                identifiers.add(
                    value
                )


        mask = pd.Series(
            False,
            index=qc.index
        )


        used_columns = []


        for column in possible_qc_ids:

            values = (
                qc[column]
                .astype(str)
                .str.strip()
            )


            this_mask = (
                values.isin(
                    identifiers
                )
            )


            if this_mask.any():

                used_columns.append(
                    column
                )


            mask |= this_mask


        matches = qc[
            mask
        ].copy()


        if len(matches) == 0:

            raise RuntimeError(
                f"{source_name}: "
                f"no QC record found for "
                f"{frozen_id!r}."
            )


        mapped_lengths = pd.to_numeric(

            matches[
                length_column
            ],

            errors="coerce"

        ).dropna().unique()


        if len(mapped_lengths) == 0:

            raise RuntimeError(
                f"{source_name}: "
                f"mapped read length missing "
                f"for {frozen_id!r}."
            )


        if len(mapped_lengths) > 1:

            raise RuntimeError(
                f"{source_name}: conflicting "
                f"mapped lengths for "
                f"{frozen_id!r}:\n"
                f"{mapped_lengths.tolist()}"
            )


        length = float(
            mapped_lengths[0]
        )


        if (
            not np.isfinite(length)
            or
            length <= 0
        ):

            raise RuntimeError(
                f"{source_name}: invalid "
                f"mapped length {length} "
                f"for {frozen_id!r}."
            )


        lengths[
            expression_column
        ] = length


        audit_rows.append(
            {
                "Frozen ID":
                    frozen_id,

                "Expression column":
                    expression_column,

                "Average mapped length":
                    length,

                "QC matched using":
                    ";".join(
                        used_columns
                    )
            }
        )


    lengths = pd.Series(
        lengths,
        dtype=float
    )


    audit_dataframe = pd.DataFrame(
        audit_rows
    )


    print(
        f"{source_name}: mapped lengths "
        f"for {len(lengths)} samples."
    )


    print(
        lengths.describe()
    )


    return (
        lengths,
        audit_dataframe
    )


tcga_lengths, tcga_qc_audit = (
    get_sample_lengths(

        tcga_mapping,

        tcga_qc,

        tcga_length_col,

        "TCGA"
    )
)


gtex_lengths, gtex_qc_audit = (
    get_sample_lengths(

        gtex_mapping,

        gtex_qc,

        gtex_length_col,

        "GTEx"
    )
)


# ======================================================================
# Save QC audit
# ======================================================================

tcga_qc_audit.to_csv(

    OUTDIR
    / "STAGE_A_TCGA_READ_LENGTH_AUDIT.csv",

    index=False
)


gtex_qc_audit.to_csv(

    OUTDIR
    / "STAGE_A_GTEX_READ_LENGTH_AUDIT.csv",

    index=False
)


# ======================================================================
# 13 — CORRECTED gene_sums STREAMING FUNCTION
# ======================================================================

section(
    "8 — LOADING SELECTED GENE EXPRESSION"
)


def stream_selected_gene_sums(
    url,
    selected_columns,
    expected_header,
    chunk_size=5000
):

    selected_columns = list(
        selected_columns
    )


    with requests.get(
        url,
        stream=True,
        timeout=600
    ) as response:

        response.raise_for_status()

        response.raw.decode_content = False


        with gzip.GzipFile(
            fileobj=response.raw,
            mode="rb"
        ) as gz:


            text = io.TextIOWrapper(
                gz,
                encoding="utf-8"
            )


            # ======================================================
            # Find actual table header
            # Skip recount3 metadata lines beginning with ##
            # ======================================================

            while True:

                header_line = text.readline()


                if header_line == "":

                    raise RuntimeError(
                        "End of gene_sums file reached "
                        "before actual header."
                    )


                header_line = (
                    header_line
                    .rstrip("\r\n")
                )


                if header_line.startswith("##"):

                    continue


                actual_header = (
                    header_line
                    .split("\t")
                )

                break


            # ======================================================
            # Validate header
            # ======================================================

            if actual_header != expected_header:

                raise RuntimeError(

                    "gene_sums header changed between "
                    "header inspection and reading.\n\n"

                    f"Expected:\n"
                    f"{expected_header[:10]}\n\n"

                    f"Actual:\n"
                    f"{actual_header[:10]}"
                )


            gene_column = (
                actual_header[0]
            )


            missing_columns = (

                set(
                    selected_columns
                )

                -

                set(
                    actual_header
                )
            )


            if missing_columns:

                raise RuntimeError(

                    f"{len(missing_columns)} "
                    "requested expression columns "
                    "are missing.\n\n"

                    f"Examples:\n"
                    f"{list(missing_columns)[:10]}"
                )


            usecols = [

                gene_column

            ] + selected_columns


            pieces = []

            total = 0


            reader = pd.read_csv(

                text,

                sep="\t",

                names=actual_header,

                header=None,

                usecols=usecols,

                chunksize=chunk_size,

                low_memory=False
            )


            for chunk in reader:

                pieces.append(
                    chunk
                )


                total += len(chunk)


                print(
                    f"Genes read: {total:,}",
                    end="\r"
                )


            print()


    result = pd.concat(

        pieces,

        ignore_index=True
    )


    result = result.set_index(
        gene_column
    )


    result = result[
        selected_columns
    ]


    return result


# ======================================================================
# TCGA expression
# ======================================================================

tcga_auc = stream_selected_gene_sums(

    tcga_gene_url,

    tcga_mapping[
        "Expression column"
    ].tolist(),

    tcga_header,

    CHUNK_SIZE
)


print(
    "\nTCGA raw coverage matrix:",
    tcga_auc.shape
)


# ======================================================================
# GTEx expression
# ======================================================================

gtex_auc = stream_selected_gene_sums(

    gtex_gene_url,

    gtex_mapping[
        "Expression column"
    ].tolist(),

    gtex_header,

    CHUNK_SIZE
)


print(
    "GTEx raw coverage matrix:",
    gtex_auc.shape
)


# ======================================================================
# 14 — GENE UNIVERSE
# ======================================================================

section(
    "9 — GENE UNIVERSE"
)


print(
    "TCGA genes:",
    tcga_auc.shape[0]
)


print(
    "GTEx genes:",
    gtex_auc.shape[0]
)


same_index = (
    tcga_auc.index.equals(
        gtex_auc.index
    )
)


print(
    "Identical GENCODE v26 gene index:",
    same_index
)


if not same_index:

    common_genes = (
        tcga_auc.index
        .intersection(
            gtex_auc.index
        )
    )


    print(
        "Common genes:",
        len(common_genes)
    )


    tcga_auc = (
        tcga_auc.loc[
            common_genes
        ]
    )


    gtex_auc = (
        gtex_auc.loc[
            common_genes
        ]
    )


# ======================================================================
# 15 — COVERAGE → ESTIMATED READ COUNTS
# ======================================================================

section(
    "10 — COVERAGE → READ COUNTS"
)


def coverage_to_reads(
    auc,
    lengths
):

    missing_lengths = (

        set(
            auc.columns
        )

        -

        set(
            lengths.index
        )
    )


    if missing_lengths:

        raise RuntimeError(
            "Mapped read lengths missing "
            "for expression samples:\n"
            f"{list(missing_lengths)[:10]}"
        )


    lengths = (
        lengths.loc[
            auc.columns
        ]
    )


    counts = (
        auc.div(
            lengths,
            axis=1
        )
    )


    counts = counts.round()


    if counts.isna().any().any():

        raise RuntimeError(
            "NaN produced during "
            "coverage-to-count conversion."
        )


    if (
        counts < 0
    ).any().any():

        raise RuntimeError(
            "Negative counts detected."
        )


    return counts.astype(
        np.int64
    )


tcga_counts = (
    coverage_to_reads(
        tcga_auc,
        tcga_lengths
    )
)


gtex_counts = (
    coverage_to_reads(
        gtex_auc,
        gtex_lengths
    )
)


print(
    "TCGA count matrix:",
    tcga_counts.shape
)

print(
    "GTEx count matrix:",
    gtex_counts.shape
)


# ======================================================================
# 16 — RENAME TO FROZEN SAMPLE IDS
# ======================================================================

tcga_expr_to_frozen = dict(

    zip(

        tcga_mapping[
            "Expression column"
        ],

        tcga_mapping[
            "Frozen ID"
        ]
    )
)


gtex_expr_to_frozen = dict(

    zip(

        gtex_mapping[
            "Expression column"
        ],

        gtex_mapping[
            "Frozen ID"
        ]
    )
)


tcga_counts = tcga_counts.rename(
    columns=tcga_expr_to_frozen
)


gtex_counts = gtex_counts.rename(
    columns=gtex_expr_to_frozen
)


tcga_counts = (
    tcga_counts[
        tumor_ids
    ]
)


gtex_counts = (
    gtex_counts[
        normal_ids
    ]
)


# ======================================================================
# 17 — SAMPLE COUNT QC
# ======================================================================

section(
    "11 — COUNT MATRIX QC"
)


print(
    "TCGA library size:"
)

print(
    tcga_counts
    .sum(axis=0)
    .describe()
)


print(
    "\nGTEx library size:"
)

print(
    gtex_counts
    .sum(axis=0)
    .describe()
)


print(
    "\nTCGA detected genes:"
)

print(
    (tcga_counts > 0)
    .sum(axis=0)
    .describe()
)


print(
    "\nGTEx detected genes:"
)

print(
    (gtex_counts > 0)
    .sum(axis=0)
    .describe()
)


# ======================================================================
# 18 — COMBINE TUMOR + NORMAL
# ======================================================================

section(
    "12 — COMBINED DISCOVERY MATRIX"
)


combined_counts = pd.concat(

    [
        tcga_counts,
        gtex_counts
    ],

    axis=1
)


if (
    combined_counts
    .columns
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Duplicate sample IDs after "
        "combining tumor and normal."
    )


condition = pd.Series(

    (
        ["tumor"] * len(tumor_ids)
    )

    +

    (
        ["normal"] * len(normal_ids)
    ),

    index=combined_counts.columns,

    name="condition"
)


print(
    "Genes:",
    combined_counts.shape[0]
)


print(
    "Samples:",
    combined_counts.shape[1]
)


print(
    "\nGroup distribution:"
)

print(
    condition.value_counts()
)


if combined_counts.shape[1] != 394:

    raise RuntimeError(
        "Expected 394 total samples "
        f"but found {combined_counts.shape[1]}."
    )


# ======================================================================
# 19 — LOW-EXPRESSION FILTER
# ======================================================================

section(
    "13 — LOW-EXPRESSION FILTER"
)


smaller_group = min(

    len(tumor_ids),

    len(normal_ids)
)


MIN_SAMPLES = math.ceil(

    smaller_group
    *
    MIN_GROUP_FRACTION
)


print(
    "Smaller group size:",
    smaller_group
)


print(
    "Minimum count:",
    MIN_COUNT
)


print(
    "Minimum samples:",
    MIN_SAMPLES
)


genes_before = (
    combined_counts.shape[0]
)


keep_genes = (

    (
        combined_counts
        >= MIN_COUNT
    )

    .sum(axis=1)

    >= MIN_SAMPLES
)


filtered_counts = (
    combined_counts.loc[
        keep_genes
    ]
    .copy()
)


print(
    "\nGenes before:",
    genes_before
)


print(
    "Genes retained:",
    filtered_counts.shape[0]
)


print(
    "Genes removed:",
    genes_before
    -
    filtered_counts.shape[0]
)


print(
    "Retention:",
    round(
        100
        *
        filtered_counts.shape[0]
        /
        genes_before,
        2
    ),
    "%"
)


# ======================================================================
# Save filtered count matrix
# ======================================================================

if SAVE_FILTERED_COUNTS:

    FILTERED_COUNTS_FILE = (

        OUTDIR
        / "STAGE_A_FILTERED_READ_COUNTS.csv.gz"
    )


    filtered_counts.to_csv(

        FILTERED_COUNTS_FILE,

        compression="gzip"
    )


    print(
        "\nFiltered counts saved:"
    )

    print(
        FILTERED_COUNTS_FILE
    )


# ======================================================================
# 20 — PyDESeq2 INPUT
# ======================================================================

section(
    "14 — PyDESeq2 INPUT"
)


# PyDESeq2:
# samples = rows
# genes = columns

counts_for_deseq = (
    filtered_counts
    .T
    .copy()
)


metadata = pd.DataFrame(

    {
        "condition":
            condition.loc[
                counts_for_deseq.index
            ].values
    },

    index=counts_for_deseq.index
)


metadata[
    "condition"
] = pd.Categorical(

    metadata[
        "condition"
    ],

    categories=[
        "normal",
        "tumor"
    ]
)


print(
    "Count matrix:",
    counts_for_deseq.shape
)


print(
    "Metadata:",
    metadata.shape
)


print(
    "\nGroups:"
)

print(
    metadata[
        "condition"
    ]
    .value_counts()
)


# ======================================================================
# 21 — RUN PyDESeq2
# ======================================================================

section(
    "15 — RUNNING PyDESeq2"
)


print(
    "Design: ~ condition"
)

print(
    "Reference group: normal"
)

print(
    "Contrast: tumor vs normal"
)


try:

    # newer PyDESeq2
    dds = DeseqDataSet(

        counts=counts_for_deseq,

        metadata=metadata,

        design="~condition",

        refit_cooks=True
    )


except TypeError:

    print(
        "Using legacy PyDESeq2 API."
    )


    dds = DeseqDataSet(

        counts=counts_for_deseq,

        metadata=metadata,

        design_factors="condition",

        refit_cooks=True
    )


dds.deseq2()


# ======================================================================
# 22 — DIFFERENTIAL EXPRESSION
# ======================================================================

section(
    "16 — DIFFERENTIAL EXPRESSION"
)


stats = DeseqStats(

    dds,

    contrast=[
        "condition",
        "tumor",
        "normal"
    ]
)


stats.summary()


results = (
    stats.results_df
    .copy()
)


# ======================================================================
# Rename columns
# ======================================================================

results = results.rename(

    columns={

        "log2FoldChange":
            "log2FC",

        "padj":
            "FDR"
    }
)


results.index.name = (
    "gene_id_v26"
)


# ======================================================================
# Strip Ensembl version
# ======================================================================

results[
    "gene_id_no_version"
] = (

    results.index
    .astype(str)
    .str.replace(
        r"\.\d+$",
        "",
        regex=True
    )
)


# ======================================================================
# 23 — DE QC
# ======================================================================

section(
    "17 — DE RESULT QC"
)


print(
    "Genes tested:",
    len(results)
)


print(
    "Valid p-values:",
    results[
        "pvalue"
    ]
    .notna()
    .sum()
)


print(
    "Valid FDR:",
    results[
        "FDR"
    ]
    .notna()
    .sum()
)


print(
    "\nlog2FC summary:"
)


print(
    results[
        "log2FC"
    ]
    .describe()
)


# ======================================================================
# 24 — SIGNIFICANT DEGs
# ======================================================================

section(
    "18 — SIGNIFICANT DEGs"
)


valid_results = (
    results[
        results[
            "FDR"
        ]
        .notna()
    ]
    .copy()
)


significant = (

    valid_results[

        (
            valid_results[
                "FDR"
            ]
            <
            FDR_THRESHOLD
        )

        &

        (
            valid_results[
                "log2FC"
            ]
            .abs()
            >=
            LOG2FC_THRESHOLD
        )
    ]

    .copy()
)


upregulated = (

    significant[
        significant[
            "log2FC"
        ]
        >
        0
    ]

    .copy()
)


downregulated = (

    significant[
        significant[
            "log2FC"
        ]
        <
        0
    ]

    .copy()
)


print(
    f"FDR threshold: {FDR_THRESHOLD}"
)


print(
    f"|log2FC| threshold: "
    f"{LOG2FC_THRESHOLD}"
)


print(
    "\nSignificant DEGs:",
    len(significant)
)


print(
    "Upregulated in GBM:",
    len(upregulated)
)


print(
    "Downregulated in GBM:",
    len(downregulated)
)


# ======================================================================
# 25 — SORT
# ======================================================================

results_sorted = (
    results
    .sort_values(

        [
            "FDR",
            "pvalue"
        ],

        na_position="last"
    )
)


significant_sorted = (
    significant
    .sort_values(

        [
            "FDR",
            "log2FC"
        ],

        ascending=[
            True,
            False
        ]
    )
)


upregulated_sorted = (
    upregulated
    .sort_values(

        [
            "FDR",
            "log2FC"
        ],

        ascending=[
            True,
            False
        ]
    )
)


downregulated_sorted = (
    downregulated
    .sort_values(

        [
            "FDR",
            "log2FC"
        ],

        ascending=[
            True,
            True
        ]
    )
)


# ======================================================================
# 26 — SAVE RESULTS
# ======================================================================

section(
    "19 — SAVING DE RESULTS"
)


FULL_RESULTS_FILE = (
    OUTDIR
    / "STAGE_A_DE_FULL_RESULTS.csv"
)


DEG_FILE = (
    OUTDIR
    / "STAGE_A_DEG_LIST.csv"
)


UP_FILE = (
    OUTDIR
    / "STAGE_A_DEG_UPREGULATED.csv"
)


DOWN_FILE = (
    OUTDIR
    / "STAGE_A_DEG_DOWNREGULATED.csv"
)


METADATA_FILE = (
    OUTDIR
    / "STAGE_A_DE_SAMPLE_METADATA.csv"
)


results_sorted.to_csv(
    FULL_RESULTS_FILE
)


significant_sorted.to_csv(
    DEG_FILE
)


upregulated_sorted.to_csv(
    UP_FILE
)


downregulated_sorted.to_csv(
    DOWN_FILE
)


metadata.to_csv(
    METADATA_FILE
)


print(
    FULL_RESULTS_FILE
)

print(
    DEG_FILE
)

print(
    UP_FILE
)

print(
    DOWN_FILE
)

print(
    METADATA_FILE
)


# ======================================================================
# 27 — FINAL SUMMARY
# ======================================================================

summary = pd.DataFrame(

    {

        "Metric": [

            "Independent GBM tumors",

            "GTEx cortical controls",

            "Total samples",

            "Genes before filtering",

            "Genes after filtering",

            "Genes with valid FDR",

            "Significant DEGs",

            "Upregulated DEGs",

            "Downregulated DEGs",

            "FDR threshold",

            "Absolute log2FC threshold",

            "Minimum count",

            "Minimum samples"
        ],


        "Value": [

            len(tumor_ids),

            len(normal_ids),

            len(tumor_ids)
            +
            len(normal_ids),

            genes_before,

            filtered_counts.shape[0],

            results[
                "FDR"
            ]
            .notna()
            .sum(),

            len(significant),

            len(upregulated),

            len(downregulated),

            FDR_THRESHOLD,

            LOG2FC_THRESHOLD,

            MIN_COUNT,

            MIN_SAMPLES
        ]
    }
)


SUMMARY_FILE = (
    OUTDIR
    / "STAGE_A_DE_SUMMARY.csv"
)


summary.to_csv(
    SUMMARY_FILE,
    index=False
)


# ======================================================================
# 28 — FINAL DISPLAY
# ======================================================================

section(
    "FINAL STAGE-A DE RESULT"
)


print(
    summary.to_string(
        index=False
    )
)


print(
    "\nTOP 20 SIGNIFICANT DEGs"
)


columns_to_show = [

    "gene_id_no_version",

    "baseMean",

    "log2FC",

    "lfcSE",

    "stat",

    "pvalue",

    "FDR"
]


available_columns = [

    c for c in columns_to_show

    if c in significant_sorted.columns
]


print(

    significant_sorted[
        available_columns
    ]

    .head(20)

    .to_string()
)


# ======================================================================
# 29 — COMPLETE
# ======================================================================

section(
    "STAGE A COMPLETE"
)


print(
    "Discovery cohort:"
)

print(
    f"GBM = {len(tumor_ids)}"
)

print(
    f"Normal cortex = {len(normal_ids)}"
)


print(
    "\nSignificant DEGs:",
    len(significant)
)


print(
    "Upregulated:",
    len(upregulated)
)


print(
    "Downregulated:",
    len(downregulated)
)


print(
    "\nResults folder:"
)

print(
    OUTDIR
)


print(
    "\nNEXT STEP:"
)

print(
    "DEGs → STRING subnetwork → "
    "patient-specific rewiring → "
    "144-patient survival modeling."
)

### STAGE B (CORRECTED) — DEG-ANCHORED STRING NETWORK -> HUB RANKING

In [ ]:
# ======================================================================
# STAGE B (CORRECTED) — DEG-ANCHORED STRING NETWORK -> HUB RANKING
# This graph is also the Stage C rewiring backbone -- built once, reused.
# ======================================================================
#
# Design decisions locked in this version (all four issues from review):
#   1. Betweenness: weighted, distance = 1/STRING_weight (stronger STRING
#      confidence -> shorter path), stored as a separate 'distance' edge
#      attribute so it never gets confused with the 'weight' attribute
#      used by PageRank/eigenvector (where higher = more important).
#   2. Eigenvector centrality: computed on the largest connected
#      component (LCC) only. Nodes outside the LCC get NaN, not 0, and
#      are excluded from that metric's percentile (pandas .rank(pct=True)
#      already does this by default -- verified below).
#   3. Isolated DEGs: NOT silently dropped. A DEG anchor that still has
#      zero high-confidence STRING neighbors even after neighbor
#      expansion is saved to an audit file and excluded from topology
#      metrics (which are undefined for a degree-0 node), not deleted.
#   4. Duplicate STRING mappings: resolved deterministically -- smallest
#      FDR first, then largest |log2FC| as tie-break. Detected and
#      counted explicitly before resolving, never silently.
#
# Graph scope: DEG anchors + their first-degree STRING neighbors (not
# DEG-DEG-only). Anchors are flagged via Is_DEG_Anchor so Stage C knows
# which nodes are the actual rewiring targets vs. contextual neighbors.

from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx

# ======================================================================
# 0 — PATHS
# ======================================================================

DOWNLOADS = BASE
DE_DIR = DOWNLOADS / "TCGA_GBM_STAGE_A_DISCOVERY" / "DE_RESULTS"
STRING_DIR = DOWNLOADS / "TCGA_GBM_STRING"
OUTDIR = DOWNLOADS / "TCGA_GBM_STAGE_B_NETWORK"
OUTDIR.mkdir(parents=True, exist_ok=True)

DEG_FILE = DE_DIR / "STAGE_A_DEG_LIST.csv"

TOP_N_HUBS = 50
BETWEENNESS_K = 500
RANDOM_SEED = 42


def section(title):
    print("\n" + "=" * 75)
    print(title)
    print("=" * 75)


def strip_ensembl_version(x):
    return str(x).split(".")[0].strip()


def find_string_files(string_dir: Path):
    """Same auto-discovery as before -- unchanged, it worked fine."""
    candidate_files = list(string_dir.glob("*.csv"))
    if not candidate_files:
        raise RuntimeError(f"No CSV files found in:\n{string_dir}")

    mapping_file, network_file = None, None
    for file in candidate_files:
        try:
            preview = pd.read_csv(file, nrows=5, low_memory=False)
        except Exception:
            continue
        cols = set(preview.columns)
        if {"ensembl_id", "string_protein_id"}.issubset(cols):
            mapping_file = file
        if {"protein1", "protein2", "combined_score"}.issubset(cols):
            network_file = file

    if mapping_file is None:
        raise RuntimeError("Could not find the TCGA<->STRING mapping CSV.")
    if network_file is None:
        raise RuntimeError("Could not find the STRING network-edge CSV.")
    return mapping_file, network_file


# ======================================================================
# 1 — LOAD DEGs
# ======================================================================
section("1 -- LOAD STAGE-A DEGs")

deg = pd.read_csv(DEG_FILE, low_memory=False)
print("DEG rows:", len(deg))
print("Columns:", deg.columns.tolist())

if "gene_id_no_version" in deg.columns:
    deg["ensembl_id"] = deg["gene_id_no_version"].astype(str).apply(strip_ensembl_version)
elif "gene_id_v26" in deg.columns:
    deg["ensembl_id"] = deg["gene_id_v26"].astype(str).apply(strip_ensembl_version)
else:
    possible = [c for c in deg.columns if "gene" in c.lower() or "ensembl" in c.lower()]
    if not possible:
        raise RuntimeError("Could not identify Ensembl gene ID column.")
    print("Using possible gene column:", possible[0])
    deg["ensembl_id"] = deg[possible[0]].astype(str).apply(strip_ensembl_version)

deg["Regulation"] = np.where(deg["log2FC"] > 0, "UP", "DOWN")
deg["abs_log2FC"] = deg["log2FC"].abs()

print("\nUnique DEG Ensembl IDs:", deg["ensembl_id"].nunique())
print(deg["Regulation"].value_counts())


# ======================================================================
# 2 — FIND STRING FILES, LOAD MAPPING
# ======================================================================
section("2 -- STRING FILES + MAPPING")

mapping_file, network_file = find_string_files(STRING_DIR)
print("Mapping file:", mapping_file)
print("Network file:", network_file)

mapping = pd.read_csv(mapping_file, low_memory=False)
mapping["ensembl_id"] = mapping["ensembl_id"].astype(str).apply(strip_ensembl_version)
print("Mapping rows:", len(mapping))


# ======================================================================
# 3 — MAP DEGs TO STRING, RESOLVE DUPLICATES DETERMINISTICALLY (fix #4)
# ======================================================================
section("3 -- MAP DEGs TO STRING (deterministic duplicate resolution)")

deg_mapped_raw = deg.merge(mapping, on="ensembl_id", how="inner")
print(f"DEGs mapped to STRING (before dedup): {len(deg_mapped_raw)} rows, "
      f"{deg_mapped_raw['ensembl_id'].nunique()} unique Ensembl IDs, "
      f"{deg_mapped_raw['string_protein_id'].nunique()} unique STRING proteins")

dup_count = deg_mapped_raw["string_protein_id"].duplicated(keep=False).sum()
print(f"\nRows sharing a STRING protein ID with >=1 other row: {dup_count}")
if dup_count > 0:
    dup_examples = (
        deg_mapped_raw[deg_mapped_raw["string_protein_id"].duplicated(keep=False)]
        .sort_values("string_protein_id")
    )
    print("Examples of duplicate mappings before resolution:")
    print(dup_examples[["ensembl_id", "string_protein_id", "FDR", "log2FC"]].head(10).to_string(index=False))

# tie-break: smallest FDR first, then largest |log2FC|
deg_mapped = (
    deg_mapped_raw
    .sort_values(["FDR", "abs_log2FC"], ascending=[True, False])
    .drop_duplicates(subset="string_protein_id", keep="first")
    .copy()
)
print(f"\nAfter deterministic resolution (min FDR, then max |log2FC|): "
      f"{deg_mapped['string_protein_id'].nunique()} unique STRING proteins")

mapping_pct = 100 * deg_mapped["ensembl_id"].nunique() / deg["ensembl_id"].nunique()
print(f"Mapping percentage: {mapping_pct:.2f}%")

deg_mapped.to_csv(OUTDIR / "STAGE_B_DEG_STRING_MAPPING.csv", index=False)


# ======================================================================
# 4 — LOAD FULL STRING NETWORK
# ======================================================================
section("4 -- LOAD FULL STRING NETWORK")

edges = pd.read_csv(network_file, low_memory=False)
all_string_nodes = set(edges["protein1"]) | set(edges["protein2"])
print(f"Full STRING backbone: {len(edges)} edges, {len(all_string_nodes)} nodes")

deg_anchor_nodes = set(deg_mapped["string_protein_id"].dropna().astype(str))
print(f"DEG anchor nodes (mapped to STRING): {len(deg_anchor_nodes)}")


# ======================================================================
# 5 — BUILD DEG-ANCHORED NETWORK: ANCHORS + FIRST-DEGREE NEIGHBORS
#     (the key change: OR instead of AND on the two endpoints)
# ======================================================================
section("5 -- BUILD DEG-ANCHORED NETWORK (anchors + first-degree neighbors)")

anchored_edges = edges[
    edges["protein1"].isin(deg_anchor_nodes) | edges["protein2"].isin(deg_anchor_nodes)
].copy()

anchored_nodes = set(anchored_edges["protein1"]) | set(anchored_edges["protein2"])
neighbor_only_nodes = anchored_nodes - deg_anchor_nodes

print(f"Anchored network edges: {len(anchored_edges)}")
print(f"Anchored network nodes: {len(anchored_nodes)} "
      f"({len(anchored_nodes) / len(all_string_nodes) * 100:.1f}% of the full {len(all_string_nodes)}-node backbone)")
print(f"  DEG anchors present with >=1 edge: {len(anchored_nodes & deg_anchor_nodes)}")
print(f"  Non-DEG first-degree neighbors pulled in: {len(neighbor_only_nodes)}")

# fix #3: isolated DEGs -- anchors with literally zero edges even after
# neighbor expansion. Audited, not silently dropped.
isolated_deg_anchors = deg_anchor_nodes - anchored_nodes
print(f"\nDEG anchors with ZERO STRING neighbors even after expansion: "
      f"{len(isolated_deg_anchors)}")

isolated_audit = deg_mapped[deg_mapped["string_protein_id"].isin(isolated_deg_anchors)].copy()
isolated_audit.to_csv(OUTDIR / "STAGE_B_ISOLATED_DEG_ANCHORS_AUDIT.csv", index=False)
print(f"Saved isolated-anchor audit -> STAGE_B_ISOLATED_DEG_ANCHORS_AUDIT.csv "
      f"({len(isolated_audit)} rows) -- excluded from topology metrics below, "
      f"not deleted from the record.")

anchored_edges.to_csv(OUTDIR / "STAGE_B_DEG_ANCHORED_STRING_EDGES.csv", index=False)


# ======================================================================
# 6 — CONSTRUCT GRAPH -- separate 'weight' (importance) and 'distance'
#     (traversal cost) edge attributes (fix #1)
# ======================================================================
section("6 -- CONSTRUCT WEIGHTED GRAPH (weight + distance both stored)")

G = nx.Graph()
for row in anchored_edges.itertuples(index=False):
    w = float(row.combined_score) / 1000.0
    G.add_edge(row.protein1, row.protein2, weight=w, distance=1.0 / w)

nx.set_node_attributes(
    G, {n: (n in deg_anchor_nodes) for n in G.nodes()}, name="Is_DEG_Anchor"
)

print(f"Graph nodes: {G.number_of_nodes()}")
print(f"Graph edges: {G.number_of_edges()}")

components = sorted(nx.connected_components(G), key=len, reverse=True)
print(f"Connected components: {len(components)}")
if components:
    print(f"Largest component (LCC): {len(components[0])} nodes "
          f"({len(components[0]) / G.number_of_nodes() * 100:.1f}% of anchored network)")


# ======================================================================
# 7 — CENTRALITY METRICS
# ======================================================================
section("7 -- CENTRALITY METRICS")

print("Degree...")
degree = dict(G.degree())
degree_centrality = nx.degree_centrality(G)

print("PageRank (weight = STRING confidence, higher = stronger)...")
pagerank = nx.pagerank(G, weight="weight", max_iter=500)

print("Weighted betweenness (distance = 1/weight, fix #1)...")
n_nodes = G.number_of_nodes()
if n_nodes <= BETWEENNESS_K:
    betweenness = nx.betweenness_centrality(G, normalized=True, weight="distance")
else:
    print(f"Using approximate betweenness, k={BETWEENNESS_K}")
    betweenness = nx.betweenness_centrality(
        G, k=BETWEENNESS_K, normalized=True, weight="distance", seed=RANDOM_SEED
    )

print("Eigenvector centrality -- LCC ONLY, NaN elsewhere (fix #2)...")
eigenvector = {node: np.nan for node in G.nodes()}
if components:
    lcc_nodes = components[0]
    G_lcc = G.subgraph(lcc_nodes)
    try:
        eig_lcc = nx.eigenvector_centrality(G_lcc, weight="weight", max_iter=2000, tol=1e-06)
        eigenvector.update(eig_lcc)
        print(f"  Computed for {len(eig_lcc)} LCC nodes; "
              f"{G.number_of_nodes() - len(eig_lcc)} non-LCC nodes left as NaN")
    except Exception as e:
        print(f"  Eigenvector centrality failed even on LCC: {e}")


# ======================================================================
# 8 — NODE TABLE
# ======================================================================
section("8 -- BUILD NODE TABLE")

node_metrics = pd.DataFrame({"string_protein_id": list(G.nodes())})
node_metrics["Is_DEG_Anchor"] = node_metrics["string_protein_id"].map(
    lambda n: G.nodes[n]["Is_DEG_Anchor"]
)
node_metrics["Degree"] = node_metrics["string_protein_id"].map(degree)
node_metrics["Degree_Centrality"] = node_metrics["string_protein_id"].map(degree_centrality)
node_metrics["Betweenness"] = node_metrics["string_protein_id"].map(betweenness)
node_metrics["PageRank"] = node_metrics["string_protein_id"].map(pagerank)
node_metrics["Eigenvector"] = node_metrics["string_protein_id"].map(eigenvector)

annotation_columns = ["string_protein_id", "ensembl_id"]
for col in ["gene_name", "preferred_name"]:
    if col in deg_mapped.columns:
        annotation_columns.append(col)
annotation_columns += ["log2FC", "FDR", "Regulation"]

annotation = deg_mapped[annotation_columns].drop_duplicates(subset="string_protein_id")
node_metrics = node_metrics.merge(annotation, on="string_protein_id", how="left")

print(f"Node table: {len(node_metrics)} rows "
      f"({node_metrics['Is_DEG_Anchor'].sum()} DEG anchors, "
      f"{(~node_metrics['Is_DEG_Anchor']).sum()} non-DEG neighbors)")


# ======================================================================
# 9 — CONSENSUS HUB SCORE (percentiles auto-exclude NaN, verified below)
# ======================================================================
section("9 -- CONSENSUS HUB RANKING")

centrality_columns = ["Degree_Centrality", "Betweenness", "PageRank", "Eigenvector"]
for col in centrality_columns:
    node_metrics[col + "_Percentile"] = node_metrics[col].rank(pct=True, method="average")

percentile_columns = [c + "_Percentile" for c in centrality_columns]
# skipna=True (pandas default for .mean()) -- a node missing only the
# Eigenvector percentile (non-LCC) still gets a consensus score from the
# other 3 metrics, rather than being forced to NaN entirely.
node_metrics["N_Metrics_Available"] = node_metrics[percentile_columns].notna().sum(axis=1)
node_metrics["Consensus_Hub_Score"] = node_metrics[percentile_columns].mean(axis=1, skipna=True)

n_partial = (node_metrics["N_Metrics_Available"] < 4).sum()
print(f"Nodes with a consensus score based on <4 metrics (non-LCC, missing eigenvector): {n_partial}")

# Report hub ranking for DEG ANCHORS ONLY -- neighbor context nodes
# shouldn't be reported as "hub DEGs" in the interpretation table.
anchors_only = node_metrics[node_metrics["Is_DEG_Anchor"]].copy()
anchors_only = anchors_only.sort_values("Consensus_Hub_Score", ascending=False).reset_index(drop=True)
anchors_only["Hub_Rank"] = np.arange(1, len(anchors_only) + 1)

top_hubs = anchors_only.head(TOP_N_HUBS).copy()

show_columns = [c for c in [
    "Hub_Rank", "gene_name", "preferred_name", "ensembl_id", "Regulation",
    "log2FC", "FDR", "Degree", "Betweenness", "PageRank", "Eigenvector",
    "N_Metrics_Available", "Consensus_Hub_Score",
] if c in top_hubs.columns]

print(f"\nTop {min(TOP_N_HUBS, len(top_hubs))} hub DEG anchors:")
print(top_hubs[show_columns].head(30).to_string(index=False))


# ======================================================================
# 10 — SAVE
# ======================================================================
section("10 -- SAVE STAGE-B RESULTS")

ALL_NODE_FILE = OUTDIR / "STAGE_B_ALL_ANCHORED_NETWORK_METRICS.csv"
TOP_HUB_FILE = OUTDIR / "STAGE_B_TOP50_HUB_DEG_ANCHORS.csv"
SUMMARY_FILE = OUTDIR / "STAGE_B_NETWORK_SUMMARY.csv"

node_metrics.to_csv(ALL_NODE_FILE, index=False)
top_hubs.to_csv(TOP_HUB_FILE, index=False)

summary = pd.DataFrame({
    "Metric": [
        "Stage-A DEGs", "Upregulated DEGs", "Downregulated DEGs",
        "DEGs mapped to STRING", "Duplicate STRING mappings resolved",
        "DEG anchor nodes", "Isolated DEG anchors (audited, excluded from topology)",
        "Non-DEG first-degree neighbors", "Anchored network total nodes",
        "Anchored network edges", "Connected components", "Largest component (LCC) size",
        "Nodes with partial (<4) consensus metrics", "Top hub anchors saved",
    ],
    "Value": [
        deg["ensembl_id"].nunique(),
        int((deg["Regulation"] == "UP").sum()),
        int((deg["Regulation"] == "DOWN").sum()),
        deg_mapped["ensembl_id"].nunique(),
        dup_count,
        len(deg_anchor_nodes),
        len(isolated_deg_anchors),
        len(neighbor_only_nodes),
        G.number_of_nodes(),
        G.number_of_edges(),
        len(components),
        len(components[0]) if components else 0,
        n_partial,
        len(top_hubs),
    ],
})
summary.to_csv(SUMMARY_FILE, index=False)
print(summary.to_string(index=False))

print("\nFiles created:")
for f in [ALL_NODE_FILE, TOP_HUB_FILE, SUMMARY_FILE,
          OUTDIR / "STAGE_B_DEG_ANCHORED_STRING_EDGES.csv",
          OUTDIR / "STAGE_B_DEG_STRING_MAPPING.csv",
          OUTDIR / "STAGE_B_ISOLATED_DEG_ANCHORS_AUDIT.csv"]:
    print(" ", f)

section("STAGE B COMPLETE")
print("This same graph (G / STAGE_B_DEG_ANCHORED_STRING_EDGES.csv) is the")
print("Stage C rewiring backbone -- rewiring is computed primarily for")
print("Is_DEG_Anchor==True nodes, using neighbor-only nodes as local context.")